# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


*   Task: To score/rank content pieces by expected engagement_rate (a continuous target)
*   Method: Random Forest Regressor (a small tree ensemble)
* Why? It can handle the mixed feature types. It is robust to missing values and gives built-in feature-importance ranking we can read. Additionally, it is simple to train for an honest model.
* Importance: It is important because we will evaluate the model as a ranker by sorting predicted engagement and reporting mean enegagement in the top K (K=20,50,100).



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

A grouped split by "client_id" is utilised: training on one set of clients and testing on different clients. Items from the same clients may share patterns and practices so splitting by clients reduces the risk of client bias rather than general signals. Therefore, it gives a more honest estimate of how well the score generalises to new clients.

The dataset is a single 90-day snapshot per piece (there is no time series).


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Plan:
* Recreate the baseline rule from week-4
* Train a RandomForestRegressor on the training clients


In [1]:
# Train + compare vs baseline (runnable)
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Load data (robust)
RAW_URL = "https://raw.githubusercontent.com/reezcon/First-ML-Pipeline/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW_URL)
df = df.copy()  # work on a copy

# Basic target and base rate
y_all = df["engagement_rate"].fillna(0).astype(float)
print("Base rate (mean engagement_rate) = {:.4f}".format(y_all.mean()))

# --- Recreate Week-4 baseline score (rule from w04) so comparison is reproducible here ---
# We use impressions_90d, ctr, avg_position presence and value to compute the same score.
df["impressions_90d"] = pd.to_numeric(df.get("impressions_90d", df.get("impressions", 0)), errors="coerce").fillna(0)
df["ctr"] = pd.to_numeric(df.get("ctr"), errors="coerce") if "ctr" in df.columns else None
df["avg_position"] = pd.to_numeric(df.get("avg_position"), errors="coerce") if "avg_position" in df.columns else None
df["avg_position_missing"] = ((df["avg_position"].isna()) | (df["avg_position"] == 0)).astype(int)

TH_IMPR_HIGH = 1000
TH_IMPR_MED = 500
TH_CTR_LOW = 0.5
TH_POSITION_POOR = 10

df["high_impr"] = (df["impressions_90d"] >= TH_IMPR_HIGH).astype(int)
df["med_impr"] = ((df["impressions_90d"] >= TH_IMPR_MED) & (df["impressions_90d"] < TH_IMPR_HIGH)).astype(int)
if df["ctr"] is not None:
    df["low_ctr"] = ((df["ctr"].notna()) & (df["ctr"] <= TH_CTR_LOW)).astype(int)
else:
    df["low_ctr"] = 0
df["poor_position"] = ((df["avg_position"].notna()) & (df["avg_position"] > TH_POSITION_POOR) & (df["avg_position_missing"] == 0)).astype(int)
df["missing_position"] = df["avg_position_missing"].astype(int)

df["baseline_score"] = 3*df["high_impr"] + 2*df["med_impr"] + 2*df["low_ctr"] + 2*df["poor_position"] + 1*df["missing_position"]

# --- Prepare features for the model ---
features = ["content_type", "position_tier", "freshness_tier", "word_count", "competition_level", "cpc", "search_volume"]
X = df[features].copy()
y = df["engagement_rate"].fillna(0).astype(float)

# Split by client_id (grouped split)
groups = df["client_id"].fillna("MISSING_CLIENT")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].reset_index(drop=True)  # keep original columns for baseline ranking

print("Train rows:", len(X_train), "Test rows:", len(X_test))

# Simple preprocessing: categorical one-hot, numeric median impute
categorical_cols = ["content_type", "position_tier", "freshness_tier", "competition_level"]
numeric_cols = ["word_count", "cpc", "search_volume"]

cat_pipe = make_pipeline(SimpleImputer(strategy="constant", fill_value="MISSING"), OneHotEncoder(handle_unknown="ignore"))
num_pipe = make_pipeline(SimpleImputer(strategy="median"))

pre = ColumnTransformer([
    ("cat", cat_pipe, categorical_cols),
    ("num", num_pipe, numeric_cols),
], remainder="drop")

# Model: RandomForestRegressor with fixed random state
model = make_pipeline(pre, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))

print("Training RandomForestRegressor ...")
model.fit(X_train, y_train)

# Predictions on test set
preds_test = model.predict(X_test)

# Create evaluation helpers: measure mean engagement in top K by a score (higher is better)
def mean_top_k_score(score_array, labels, k):
    order = np.argsort(-np.asarray(score_array))
    topk = order[:k]
    return np.asarray(labels).astype(float)[topk].mean()

# Evaluate base rate on test
base_test = y_test.mean()
print("Base rate on TEST = {:.4f}".format(base_test))

# Baseline ranking: use baseline_score computed earlier for df_test
baseline_scores_test = df_test["baseline_score"].values
# Model ranking: align preds_test with df_test rows (X_test is in same index order as df_test)
# But to be safe, ensure ordering matches:
assert len(preds_test) == len(df_test)

for k in (20, 50, 100):
    m_baseline = mean_top_k_score(baseline_scores_test, df_test["engagement_rate"].values, k)
    m_model = mean_top_k_score(preds_test, df_test["engagement_rate"].values, k)
    print(f"Mean engagement in top {k} — baseline: {m_baseline:.4f} | model: {m_model:.4f}")

Base rate (mean engagement_rate) = 2.5345
Train rows: 23837 Test rows: 6163
Training RandomForestRegressor ...
Base rate on TEST = 2.9117
Mean engagement in top 20 — baseline: 2.3415 | model: 0.3780
Mean engagement in top 50 — baseline: 2.9996 | model: 1.5798
Mean engagement in top 100 — baseline: 3.6169 | model: 2.5390


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.